# S07b - Quarterly earnings breadth from SEC EDGAR filings (company fundamentals, late-data)

> **Legacy outputs retained:** At this source-only correction, the saved outputs below have not been rerun under `matched-v1`. They are historical evidence, not corrected results. Rerun the evaluation before drawing conclusions, then refresh this notice and the interpretation.

**Evaluation correction: `matched-v1`.** Only this notebook opts into common-window development
selection, matched-sample placebos, and a dev+val-only DSR acceptance gate. Future registry writes
use `S07b-matched-v1`, preserving legacy `S07b` rows; unchanged variants still count once toward
trial multiplicity because their strategy/parameter identities are unchanged.

**Hypothesis.** As S07: when a falling share of S&P 500 companies report year-on-year earnings growth,
the earnings cycle is turning down and equity risk is elevated (Kothari, Lewellen & Warner 2006).
S07b adds the *acceleration* of earnings growth, which turns earlier than the growth level.

**What changed vs S07 (new trial, S07 untouched)**

| | S07 | S07b |
|---|---|---|
| 10-K event | fiscal year vs prior fiscal year (about 4x a quarter in dollars) | **Q4** vs prior-year Q4 |
| Q4 source | - | 3-month Q4 tagged in the 10-K if present, else fiscal year - 9-month YTD from the same year's Q3 10-Q (both years), else no event |
| staleness | stated 135 days, **not applied** (seconds vs microseconds in the comparison) | 135 days, applied |
| variants | breadth50, agg_delta | breadth50, agg_delta, **accel50** |

| Variant | Risk-on (hold S&P 500) at month-end t when ... |
|---|---|
| breadth50 | > 50% of index members with a fresh filing report quarterly net income above the same quarter a year earlier |
| agg_delta | the sum over members of (quarterly net income - same quarter a year earlier) is > 0 |
| accel50 | > 50% of members with a fresh filing report a YoY change above their previous quarter's YoY change |

**Data and point-in-time handling**

| Item | Source | Point-in-time rule |
|---|---|---|
| Net income (us-gaap `NetIncomeLoss`), 3-month, 9-month YTD and fiscal-year facts from 10-Q / 10-K | SEC EDGAR XBRL `companyconcept` API | each event compares a quarter with the same quarter a year earlier **as reported in that filing** (derived Q4: the 10-K and the latest Q3 10-Q filed no later than it); usable from filing date + 1 day; ignored if older than 135 days |
| Acceleration | same events | compared with the same company's latest event available strictly earlier whose quarter ended 75-105 days before |
| Index membership by date | `fja05680/sp500` (MIT), snapshots 1996-2026 | members as of the decision date |
| Ticker -> CIK | SEC `company_tickers.json` (current tickers only) | static map |

**Survivorship.** As S07: only ~71% of the tickers that were members since 2008 map to a CIK today, so
early-sample breadth is measured on survivors. XBRL starts in 2009, hence the **late-data protocol**:
dev = data start..2013, val 2014-2017, holdout 2018+, separate leaderboard, weaker evidence.

**Sources.** SEC EDGAR APIs (https://www.sec.gov/edgar/sec-api-documentation; fair-access policy
https://www.sec.gov/os/accessing-edgar-data); Kothari, Lewellen & Warner (2006), *Stock returns,
aggregate earnings surprises, and behavioral finance*, JFE, SSRN 380127.

> Decision-support research only; never places orders.

In [36]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display

_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()), Path.cwd())
sys.path.insert(0, str(_root / "notebooks" / "backtest"))
import bt_common as bc  # noqa: E402
import bt_long  # noqa: E402
import bt_periods as bp  # noqa: E402
import bt_pit  # noqa: E402
import bt_registry as br  # noqa: E402
import bt_stats as bs  # noqa: E402
import seed_registry  # noqa: E402
import stage2 as s2  # noqa: E402

bc.setup_plotly("cdn")
pd.set_option("display.width", 180)
TRIAL_ID, STRATEGY, NOTEBOOK = 'S07b-matched-v1', 'edgar_earnings_breadth_q', 'fundamentals/edgar_earnings_breadth_q.ipynb'
PROTOCOL, REBALANCE = 'late', 'monthly'

sample = bt_long.long_sample()
px, cash = sample["px"], sample["cash"]
bench = bp.benchmark_returns(px, cash)  # SMA200 monthly / B&H / 60-40 / T-bills, frozen settings
print("prices", px.index[0].date(), "->", px.index[-1].date(), "| registry trials so far:", br.trial_count())
EVIDENCE_VERSION = 'matched-v1'

prices 1934-02-01 -> 2026-11-13 | registry trials so far: 37


## 1. Data and point-in-time inputs

### 1.1 Index membership and ticker -> CIK

S&P 500 members by date come from the `fja05680/sp500` snapshots. SEC data is keyed by CIK (the SEC
company id), so every ticker that was a member since 2008 is mapped to a CIK with SEC's
`company_tickers.json`. That file lists **current** tickers only, so acquired or delisted members
cannot be mapped (survivorship, see caveats).

In [37]:
sys.path.insert(0, str(_root / "notebooks" / "backtest" / "fundamentals"))
import edgar_data as ed  # noqa: E402

SINCE = "2008-01-01"
members = ed.load_membership()  # Series: snapshot date -> list of tickers
cmap = ed.ticker_cik_map()  # ticker -> CIK (current tickers only)
tickers = sorted({t for lst in members[members.index >= SINCE] for t in lst})
ciks = ed.member_ciks(SINCE)
print(f"membership snapshots: {len(members):,}, {members.index[0].date()} -> {members.index[-1].date()}")
print(f"tickers that were members since {SINCE[:4]}: {len(tickers)} | map to a CIK today: "
      f"{sum(t in cmap for t in tickers)} ({sum(t in cmap for t in tickers) / len(tickers):.0%}) -> {len(ciks)} companies")
print("unmapped examples:", ", ".join([t for t in tickers if t not in cmap][:15]))

membership snapshots: 2,720, 1996-01-02 -> 2026-08-18
tickers that were members since 2008: 899 | map to a CIK today: 641 (71%) -> 637 companies
unmapped examples: AABA, ABC, ABI, ABKFQ, ABMD, ACAS, ACS, ADS, AET, AGN, AKS, ALTR, ALXN, ANDV, ANRZQ


### 1.2 Net-income facts from SEC EDGAR

For each CIK the XBRL `companyconcept` API returns every `NetIncomeLoss` value the company has filed
(one file per company, cached under `.cache/notebooks/edgar/`; only missing or >30-day-old files are
downloaded, at ~5 requests/s per SEC fair-access rules). Kept: 10-Q / 10-K (and amendments) facts
with a start date. One filing contains several facts: the current period, the prior-year
comparative, year-to-date figures and, in many 10-Ks before 2021, a table of quarters.

In [38]:
facts = ed.load_facts(ciks, progress=False)
kind = pd.Series(np.select([facts["days"].between(80, 100), facts["days"].between(260, 285),
                            facts["days"].between(350, 380)], ["3 months", "9 months YTD", "fiscal year"], "other"),
                 index=facts.index, name="period length")
print(f"{len(facts):,} facts | {facts['cik'].nunique()} companies | {facts['accn'].nunique():,} filings | "
      f"filed {facts['filed'].min().date()} -> {facts['filed'].max().date()}")
display(pd.crosstab(facts["form"], kind))

EXAMPLE = cmap["MSFT"]
ex = facts[(facts["cik"] == EXAMPLE) & (facts["form"] == "10-K") & (facts["filed"].dt.year == 2016)]
print(f"Example - every NetIncomeLoss fact in Microsoft's 10-K filed {ex['filed'].iloc[0].date()} (fiscal year ends June):")
display(ex.assign(kind=kind).sort_values(["days", "end"])[["start", "end", "days", "kind", "val"]])

135,176 facts | 586 companies | 31,471 filings | filed 2009-04-15 -> 2026-11-06


period length,3 months,9 months YTD,fiscal year,other
form,,,,
10-K,27381,16,22970,254
10-K/A,432,7,318,15
10-Q,51336,15214,223,16252
10-Q/A,432,73,1,252


Example - every NetIncomeLoss fact in Microsoft's 10-K filed 2016-07-28 (fiscal year ends June):


,start,end,days,kind,val
53651,2015-01-01,2015-03-31,89,3 months,4985000000
53656,2015-04-01,2015-06-30,90,3 months,-3195000000
53672,2016-01-01,2016-03-31,90,3 months,3756000000
53678,2016-04-01,2016-06-30,90,3 months,3122000000
53639,2014-07-01,2014-09-30,91,3 months,4540000000
53645,2014-10-01,2014-12-31,91,3 months,5863000000
53659,2015-07-01,2015-09-30,91,3 months,4902000000
53666,2015-10-01,2015-12-31,91,3 months,5018000000
53633,2013-07-01,2014-06-30,364,fiscal year,22074000000
53653,2014-07-01,2015-06-30,364,fiscal year,12193000000


### 1.3 One quarterly year-on-year event per filing

* **10-Q**: the latest 3-month quarter vs the same quarter a year earlier, both as reported in that 10-Q.
* **10-K**: Q4 vs prior-year Q4. The 3-month Q4 pair tagged in the 10-K is used when present
  (`q4_reported`); otherwise Q4 = fiscal year - 9-month YTD for both years, the 9-month figures taken
  from the latest Q3 10-Q of the same fiscal year filed no later than the 10-K (`q4_derived`);
  otherwise the 10-K gives no event.
* Every event becomes usable on filing date + 1 day (`avail`, the index).

In [39]:
events = ed.quarterly_yoy_events(facts)
n10k = facts.loc[facts["form"].str.startswith("10-K"), "accn"].nunique()
print(f"{len(events):,} events | {events['cik'].nunique()} companies | {events.index.min().date()} -> {events.index.max().date()}")
print(f"10-K filings: {n10k:,} -> Q4 events: {(events['basis'] != 'quarter').sum():,} (rest: no tagged Q4 and no usable Q3 10-Q)")
display(pd.crosstab(events.index.year, events["basis"]).T)
print("Example - Microsoft events 2015-2017:")
display(events[events["cik"] == EXAMPLE].loc["2015":"2017"])

31,086 events | 585 companies | 2009-04-16 -> 2026-11-05
10-K filings: 7,816 -> Q4 events: 7,555 (rest: no tagged Q4 and no usable Q3 10-Q)


row_0,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026
basis,,,,,,,,,,,,,,,,,,
q4_derived,10,202,201,132,108,113,122,129,126,138,136,146,272,442,469,464,487,471
q4_reported,0,20,172,269,300,302,307,316,317,330,331,332,213,49,32,44,32,21
quarter,403,946,1216,1232,1244,1280,1305,1358,1389,1389,1445,1465,1496,1537,1546,1566,1613,1101


Example - Microsoft events 2015-2017:


,cik,filed,period_end,form,ni,ni_prior,basis
avail,,,,,,,
2015-01-27,789019,2015-01-26,2014-12-31,10-Q,5.863000e+09,6.558000e+09,quarter
2015-04-24,789019,2015-04-23,2015-03-31,10-Q,4.985000e+09,5.660000e+09,quarter
2015-08-01,789019,2015-07-31,2015-06-30,10-K,-3.195000e+09,4.612000e+09,q4_reported
2015-10-23,789019,2015-10-22,2015-09-30,10-Q,4.620000e+09,4.540000e+09,quarter
2016-01-29,789019,2016-01-28,2015-12-31,10-Q,4.998000e+09,5.863000e+09,quarter
2016-04-22,789019,2016-04-21,2016-03-31,10-Q,3.756000e+09,4.985000e+09,quarter
2016-07-29,789019,2016-07-28,2016-06-30,10-K,3.122000e+09,-3.195000e+09,q4_reported
2016-10-21,789019,2016-10-20,2016-09-30,10-Q,4.690000e+09,4.902000e+09,quarter
2017-01-27,789019,2017-01-26,2016-12-31,10-Q,5.200000e+09,5.018000e+09,quarter


### 1.4 Check: derived Q4 vs the Q4 tagged in the 10-K

Where a 10-K tags Q4 *and* a Q3 10-Q exists, both versions can be computed. Agreement shows that
`fiscal year - 9-month YTD` is reliable for the post-2021 filings where it is the only source. When
they disagree the tagged figure is used (agreed rule); the largest disagreements are listed.

In [40]:
q4check = ed.validate_q4(facts)
print(f"{len(q4check):,} 10-Ks with both | equal within 1% (or $1m) on both years: {q4check['match'].mean():.1%} | "
      f"current year {q4check['match_cur'].mean():.1%} | prior year {q4check['match_prior'].mean():.1%} | "
      f"same YoY direction: {q4check['same_direction'].mean():.1%}")
gap = (q4check["der_cur"] - q4check["rep_cur"]).abs() / q4check["rep_cur"].abs().clip(lower=1e6)
display(q4check.assign(rel_gap=gap).loc[~q4check["match"]].nlargest(8, "rel_gap")
        [["cik", "fy_end", "rep_cur", "der_cur", "rep_prior", "der_prior", "same_direction"]])

3,315 10-Ks with both | equal within 1% (or $1m) on both years: 97.3% | current year 97.6% | prior year 98.4% | same YoY direction: 99.8%


,cik,fy_end,rep_cur,der_cur,rep_prior,der_prior,same_direction
1383,793952,2012-12-31,70600.0,70639000.0,105600.0,105680000.0,True
2823,1297996,2012-12-31,54566000.0,366102000.0,45716000.0,266814000.0,True
2457,1097864,2011-12-31,-8800000.0,-42500000.0,61000000.0,61000000.0,True
3272,1674101,2020-12-31,40500000.0,-68800000.0,-33900000.0,-33900000.0,False
2460,1097864,2014-12-31,-600000.0,1700000.0,37800000.0,28300000.0,True
1140,715957,2024-12-31,145000000.0,-76000000.0,331000000.0,310000000.0,True
525,59558,2022-12-31,21000000.0,-8000000.0,708000000.0,702000000.0,True
3187,1564708,2016-06-30,-378000000.0,90000000.0,90000000.0,-378000000.0,False


### 1.5 Acceleration, saved event file and signal inputs

`accel` (used by the `accel50` variant) = 1 when an event's YoY change (`ni - ni_prior`) is above the
same company's previous-quarter YoY change. The previous quarter is the latest event available
strictly earlier whose quarter ended 75-105 days before. It is recomputed inside `build_signal`
(section 2), so the look-ahead test covers it; it is shown here only for inspection.

In [41]:
acc = ed.add_acceleration(events)
print(f"acceleration defined for {acc['accel'].notna().mean():.1%} of events; improving in {acc['accel'].mean():.1%} of those")
display(acc[acc["cik"] == EXAMPLE].loc["2015":"2017", ["period_end", "form", "basis", "ni", "ni_prior", "accel"]])

events.to_pickle(ed.EDGAR_DIR / ed.EVENTS_PKL["quarterly"])
print("saved", ed.EDGAR_DIR / ed.EVENTS_PKL["quarterly"])
inputs = {"px": px, "events": events, "members": members, "cmap": cmap}

acceleration defined for 96.6% of events; improving in 50.5% of those


,period_end,form,basis,ni,ni_prior,accel
avail,,,,,,
2015-01-27,2014-12-31,10-Q,quarter,5.863000e+09,6.558000e+09,1.0
2015-04-24,2015-03-31,10-Q,quarter,4.985000e+09,5.660000e+09,1.0
2015-08-01,2015-06-30,10-K,q4_reported,-3.195000e+09,4.612000e+09,0.0
2015-10-23,2015-09-30,10-Q,quarter,4.620000e+09,4.540000e+09,1.0
2016-01-29,2015-12-31,10-Q,quarter,4.998000e+09,5.863000e+09,0.0
2016-04-22,2016-03-31,10-Q,quarter,3.756000e+09,4.985000e+09,0.0
2016-07-29,2016-06-30,10-K,q4_reported,3.122000e+09,-3.195000e+09,1.0
2016-10-21,2016-09-30,10-Q,quarter,4.690000e+09,4.902000e+09,0.0
2017-01-27,2016-12-31,10-Q,quarter,5.200000e+09,5.018000e+09,1.0


saved C:\Users\wn686\OneDrive - Deutsche Börse AG\Desktop\REPOs\investment-firm-agents\.cache\notebooks\edgar\ni_events_q.pkl


## 2. Signal and variant grid (fixed before any backtest)

In [42]:
STALE_DAYS = 135
COVERAGE_MIN = 0.30
MEASURES = {"breadth": "grow", "agg_delta": "delta", "accel": "accel"}


def _days(ts):
    # unit-safe day numbers: px.index is datetime64[s], EDGAR dates are datetime64[us]
    return np.asarray((pd.DatetimeIndex(ts) - pd.Timestamp("1970-01-01")) / pd.Timedelta(days=1), dtype=float)


def _latest(df, col, dec):
    sub = df.dropna(subset=[col]).drop_duplicates(["avail", "cik"], keep="last")
    wide = {}
    for c in (col, "t"):
        w = sub.pivot(index="avail", columns="cik", values=c)
        wide[c] = w.reindex(w.index.union(dec)).ffill().reindex(dec)
    return wide[col], wide["t"]


def breadth_panel(inputs):
    idx = inputs["px"].index
    dec = idx[bc.check_days(idx, "monthly").to_numpy()]
    ev = ed.add_acceleration(inputs["events"])
    df = pd.DataFrame({"cik": ev["cik"].to_numpy(), "avail": ev.index,
                       "grow": (ev["ni"] > ev["ni_prior"]).astype(float).to_numpy(),
                       "delta": (ev["ni"] - ev["ni_prior"]).to_numpy(),
                       "accel": ev["accel"].to_numpy()})
    df["t"] = _days(df["avail"])
    mem = inputs["members"]
    mem = mem.reindex(mem.index.union(dec)).ffill().reindex(dec)
    n_mem = np.array([len(m) if isinstance(m, list) else 0 for m in mem])
    mem_ciks = [[inputs["cmap"].get(x) for x in m] if isinstance(m, list) else [] for m in mem]
    now = _days(dec)[:, None]
    panel = {}
    for name, col in MEASURES.items():
        val, t = _latest(df, col, dec)
        is_mem = np.array([np.isin(val.columns, m) for m in mem_ciks])
        ok = ((now - t.to_numpy()) <= STALE_DAYS) & is_mem
        v = val.where(ok)
        panel[name] = v.sum(axis=1, min_count=1) if name == "agg_delta" else v.mean(axis=1)
        panel[f"cov_{name}"] = v.notna().sum(axis=1) / np.maximum(n_mem, 1)
    return pd.DataFrame(panel, index=dec)


def build_signal(inputs, measure="breadth"):
    p = breadth_panel(inputs)
    p = p[p[f"cov_{measure}"] >= COVERAGE_MIN]  # need a meaningful share of members with fresh filings
    raw = (p[measure] > 0) if measure == "agg_delta" else (p[measure] > 0.5)
    sig = raw.astype(float)
    idx = inputs["px"].index
    return sig.reindex(sig.index.union(idx)).ffill().reindex(idx)


VARIANTS = {"breadth50": dict(measure="breadth"), "agg_delta": dict(measure="agg_delta"),
            "accel50": dict(measure="accel")}
_BASE = ("EDGAR NetIncomeLoss quarterly YoY per filing (10-K->Q4: tagged 3m else FY-9M YTD), "
         "avail filed+1d, stale>135d dropped, coverage>=30%; monthly; next_close; 5bp")
PARAMS_TEXT = {"breadth50": f"breadth50; {_BASE}", "agg_delta": f"agg_delta; {_BASE}",
               "accel50": f"accel50 (YoY change > previous quarter's YoY change); {_BASE}"}

In [43]:
panel = breadth_panel(inputs)
fig = go.Figure()
fig.add_trace(go.Scatter(x=panel.index, y=panel["breadth"], name="growth breadth (YoY)"))
fig.add_trace(go.Scatter(x=panel.index, y=panel["accel"], name="acceleration breadth"))
fig.add_trace(go.Scatter(x=panel.index, y=panel["cov_breadth"], name="coverage of members", line=dict(dash="dot")))
fig.add_hline(y=0.5, line_dash="dash", line_color="red")
fig.update_layout(title="Share of members with quarterly YoY net-income growth / acceleration (fresh filings) and coverage",
                  yaxis_tickformat=".0%", height=380, xaxis_range=["2009-01-01", str(panel.index[-1].date())])
fig.show()
display(panel.dropna(subset=["breadth"]).tail(4).round(3))

,breadth,cov_breadth,agg_delta,cov_agg_delta,accel,cov_accel
Date,,,,,,
2026-07-31,0.718,0.831,2.292353e+11,0.831,0.599,0.823
2026-08-31,0.715,0.845,2.927141e+11,0.845,0.538,0.831
2026-09-30,0.721,0.797,2.961700e+11,0.797,0.530,0.783
2026-10-30,0.714,0.791,3.010716e+11,0.791,0.512,0.777


## 3. Look-ahead checks

Every variant is recomputed with all inputs cut at past dates (monthly inputs: completed months only) and must equal the full-data signal on those dates (`stage2.truncation_test`). No backtest runs unless all pass.

In [44]:
N_CUTS = 15
trunc = {}
for name, params in VARIANTS.items():
    trunc[name] = s2.truncation_test(build_signal, inputs, n_cuts=N_CUTS, **params)
display(pd.DataFrame({k: v.row() for k, v in trunc.items()}).T)
bad = [k for k, v in trunc.items() if not v.passed]
assert not bad, f"look-ahead detected in variants {bad}: fix before any backtest"
print("all variants pass the truncation look-ahead test")

,cuts,mismatches,max_abs_diff,passed
breadth50,15,0,0.0,True
agg_delta,15,0,0.0,True
accel50,15,0,0.0,True


all variants pass the truncation look-ahead test


## 4. Development-period sweep

Each variant is scored on the **dev** period only and registered as a trial. The best dev Sharpe is chosen; validation and holdout results are never used for the choice.

**matched-v1:** score variants and benchmarks on common valid development dates, retaining their execution paths (no boundary resets).

In [45]:
signals = {name: build_signal(inputs, **params) for name, params in VARIANTS.items()}
sweep, CHOSEN = s2.dev_sweep(px, cash, bench, signals, PARAMS_TEXT, trial_id=TRIAL_ID, strategy=STRATEGY,
                             notebook=NOTEBOOK, protocol=PROTOCOL, rebalance=REBALANCE, evidence_version=EVIDENCE_VERSION)
display(sweep)
print("chosen on dev Sharpe only:", CHOSEN, "|", PARAMS_TEXT[CHOSEN])

TypeError: dev_sweep() got an unexpected keyword argument 'evidence_version'

## 5. Frozen evaluation of the chosen variant

Frozen primary metric (`PLAN.md` section 2): beat SMA200 and buy & hold in dev **and** val, one-sided block-bootstrap p <= 0.10 vs SMA200 in both, deflated Sharpe >= 0.95 with the registry trial count. Holdout is confirmatory only.

**matched-v1:** thresholds and the DSR estimator are unchanged; the DSR acceptance sample is dev + val only. Full-sample DSR is descriptive, not a gate. This is distinct from the separate `bt_stats_v2` estimator. Previously inspected holdout data do not become an untouched holdout merely by correcting this code.

In [ ]:
signal = signals[CHOSEN]
runs, cmp, stats, res = s2.evaluate(px, cash, bench, signal, protocol=PROTOCOL, rebalance=REBALANCE, evidence_version=EVIDENCE_VERSION)
display(cmp.round(3))
display(stats)
display(res.to_frame().round(3))
print(f"n_trials={res.n_trials} | DSR={res.dsr:.3f} | passes={res.passes} | holdout confirms={res.holdout_confirms}")
display(pd.Series(res.checks, name="check"))
print(f"Evidence: {EVIDENCE_VERSION}; gated DSR uses dev+val only; descriptive full DSR={res.dsr_full:.3f}")

In [ ]:
s2.plot_growth(runs, bench, f'{TRIAL_ID} {CHOSEN}', PROTOCOL).show()

## 6. Robustness: costs, placebo, peek test, decades

**matched-v1:** real and placebo paths use identical executed dates and signal availability; equity probability is calibrated on executed exposure. The legacy persistence approximation remains; exact turnover matching is not claimed.

In [ ]:
cs = bs.cost_sensitivity(px, signal, cash, rebalance=REBALANCE)
display(cs.round(3))
sma_full = bs.excess_sharpe(bench["sma200"].reindex(runs["full"].index), bench["tbill"].reindex(runs["full"].index))
be = bs.breakeven_cost_bps(px, signal, cash, bench_sharpe=sma_full, rebalance=REBALANCE)
print(f"break-even one-way cost vs SMA200 Sharpe over the same window ({sma_full:.3f}): {be:.1f} bp")
pl = bs.placebo_test(px, signal, cash, n_placebo=200, rebalance=REBALANCE, rng=np.random.default_rng(bs.SEED), evidence_version=EVIDENCE_VERSION)
display(pd.Series(pl, name="placebo").to_frame())
peek = bt_pit.lookahead_test(px, signal, cash, rebalance=REBALANCE)
print(f"peek test: Sharpe by shift {peek.sharpe_by_shift.round(3).to_dict()} | gain {peek.peek_gain:+.3f}")
display(s2.decade_table(runs["full"], bench).style.format("{:.2f}"))

## 7. Register the result

In [ ]:
note = ("passes" if res.passes else "fails") + f" {EVIDENCE_VERSION} metric (DSR dev+val); chosen {CHOSEN} on dev; protocol {PROTOCOL}"
rows = s2.register_final(TRIAL_ID, STRATEGY, NOTEBOOK, PARAMS_TEXT[CHOSEN], res, note=note, header=seed_registry.HEADER)
display(rows)
print("registry trial count:", br.trial_count())

## 8. Current signal

In [ ]:
last = signal.dropna()
print(f"latest signal ({CHOSEN}) on {last.index[-1].date()}: equity weight {last.iloc[-1]:.2f}")
chk = bc.check_days(px.index, REBALANCE)
display(last[chk.reindex(last.index).fillna(False)].tail(6).to_frame("equity weight at decision"))
print("Research output only -- not a recommendation or an order.")

## 9. Evidence summary (computed)

In [ ]:
for line in s2.evidence_lines(res, trunc[CHOSEN], peek, pl, be):
    print("*", line)

## 10. Caveats

* Survivorship: delisted / acquired members cannot be mapped to a CIK, so early breadth is measured
  on companies that survived to 2026 (an upward bias in measured breadth). Current-vintage tickers can
  also map a recycled ticker to the wrong company (AUDIT A7-02).
* Q4 follows the agreed rule *reported first*: in ~2.7% of 10-Ks that tag Q4 the reported and derived
  figures differ by more than 1%, mostly from tagging errors in the reported fact (e.g. a value tagged
  in thousands); the growth direction still agrees in 99.8%.
* 10-Ks without a tagged Q4 and without a matching Q3 10-Q (fiscal-year changes, first filings) give
  no event; an amended or late filing about an older period still becomes the company's latest event.
* Only `NetIncomeLoss` is used; companies reporting under other tags drop out. One-off items and
  losses make net-income growth noisy; `agg_delta` is dominated by the largest companies.
* Very short history (one recession in sample, 2020): weak evidence by construction. This is the
  second specification tested on the same data (after S07), so the registry trial count rises accordingly.

## 11. Interpretation

> **Legacy interpretation:** Preserved from the earlier evaluation. Its chosen variant, DSR, and performance conclusions have not been recomputed under `matched-v1`.

**S07b fails the frozen primary metric.** The dev sweep picked `accel50` (dev Sharpe 1.00 vs SMA200 0.81 and
buy & hold 0.95), but the dev edge over SMA200 is not significant (p 0.365). In validation (2014-2017) it falls to
0.68, below both SMA200 (0.77) and buy & hold (0.97; p vs SMA200 0.49). Acceleration breadth sits close to its 50%
threshold, so the signal switches about 3 times a year in validation and was out of the market about 40% of a strong
2014-17 bull market. The holdout (2018+) is better (0.69 vs SMA200 0.53, buy & hold 0.66), but holdout is
confirmatory only. Deflated Sharpe 0.78 with 37 registered trials.

**What the data changes did.** Every 10-K event is now a Q4 comparison (3,387 tagged in the 10-K, 4,168 derived as
fiscal year minus 9-month YTD; derived equals tagged within 1% for 97.3% of the 3,315 10-Ks that have both). With the
135-day staleness rule now applied (S07 compared seconds with microseconds, so it never dropped stale filings; that
changed S07's breadth50 signal in 2 of 208 months), `breadth50` improves in dev from 0.83 (S07) to 0.96 and `agg_delta`
stays weak (0.68). `breadth50` was not chosen, so its validation result is not evaluated (choice is on dev only).

**Reproducibility.** The ticker -> CIK map is SEC's *current* `company_tickers.json`, refreshed every 7 days. Between
the 2026-10-25 and 2026-11-12 runs two long-filing companies left that list and two tickers were reassigned, so they
dropped out of (or entered) the whole history: validation Sharpe moved from 0.65 to 0.68 and holdout from 0.70 to 0.69.
The verdict did not change. Pinning a dated copy of the map would make reruns identical.

**Reading.** Quarterly earnings breadth from filings is a plausible risk-off indicator, but on 2009-2026 data neither
the level nor the acceleration version beats a 200-day moving-average rule out of sample. The sample has one
recession (2020); evidence is weak by construction.